<a href="https://colab.research.google.com/github/vkebut/ANN_checkpoint/blob/main/Climate_Data_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ==============================
# 2. LOAD DATA
# ==============================
# Replace with your file path
df = pd.read_csv("climate_africa.csv")

print(df.head())
print(df.info())

# ==============================
# 3. DATA CLEANING
# ==============================
# Handle missing values
df = df.dropna()

# Ensure correct types
df['year'] = df['year'].astype(int)
df['temperature'] = df['temperature'].astype(float)

# ==============================
# 4. EXPLORATORY DATA ANALYSIS
# ==============================

# Average temperature per year
yearly_avg = df.groupby('year')['temperature'].mean().reset_index()

plt.figure()
plt.plot(yearly_avg['year'], yearly_avg['temperature'])
plt.title("Average Temperature Trend in Africa")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.grid()
plt.show()

# Country comparison
plt.figure()
sns.lineplot(data=df, x='year', y='temperature', hue='country')
plt.title("Temperature Trends by Country")
plt.show()

# ==============================
# 5. TREND ANALYSIS (REGRESSION)
# ==============================

# Predict temperature trend for each country
results = []

for country in df['country'].unique():
    temp_df = df[df['country'] == country]

    X = temp_df[['year']]
    y = temp_df['temperature']

    model = LinearRegression()
    model.fit(X, y)

    trend = model.coef_[0]  # slope

    results.append((country, trend))

trend_df = pd.DataFrame(results, columns=['country', 'warming_rate'])
print("\nWarming Trends:")
print(trend_df.sort_values(by='warming_rate', ascending=False))

# ==============================
# 6. CLUSTERING COUNTRIES
# ==============================

# Average temperature per country
country_avg = df.groupby('country')['temperature'].mean().reset_index()

scaler = StandardScaler()
scaled_data = scaler.fit_transform(country_avg[['temperature']])

kmeans = KMeans(n_clusters=3, random_state=42)
country_avg['cluster'] = kmeans.fit_predict(scaled_data)

print("\nCountry Clusters:")
print(country_avg)

# ==============================
# 7. ANOMALY DETECTION
# ==============================

# Rolling average to detect anomalies
df['rolling_mean'] = df.groupby('country')['temperature'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean()
)

df['anomaly'] = df['temperature'] - df['rolling_mean']

# Plot anomalies for one country
country_example = df['country'].unique()[0]
temp_df = df[df['country'] == country_example]

plt.figure()
plt.plot(temp_df['year'], temp_df['temperature'], label='Actual')
plt.plot(temp_df['year'], temp_df['rolling_mean'], label='Rolling Mean')
plt.title(f"Anomaly Detection - {country_example}")
plt.legend()
plt.show()

# ==============================
# 8. SAVE RESULTS FOR PAPER
# ==============================

trend_df.to_csv("warming_trends.csv", index=False)
country_avg.to_csv("country_clusters.csv", index=False)

print("\nAnalysis complete. Files saved.")